# Public-trace replay with cache pressure
Select T4, run all and upload the updated gpu-serving-source.zip. The ZIP includes the already-downloaded bounded metadata sample; this notebook does not download the dataset again.

A fixed uniform sample of 24 requests spans the 403 chronological source records. It retains 6 prefix identities. At 96 MiB, two 1,024-token prefixes fit. An offline LRU check confirms 5 evictions and one request returning after eviction. This is a small pressure pilot, not broad policy validation.

No-cache, LRU, LFU and learned policies each run twice, plus one correctness oracle: 9 runs / 216 request executions. Output lengths follow source counts capped at 64. Prompt tokens are surrogates derived from public block IDs, inputs are capped, original arrival gaps are not replayed, and omitted requests change reuse distances. No production speedup is assumed.


In [ ]:
import os, sys, subprocess, zipfile, hashlib, json
from pathlib import Path
from google.colab import files
uploaded = files.upload()  # Select gpu-serving-source.zip
repo = Path("/content/kv-optimization-upload")
repo.mkdir(exist_ok=True)
source = next((name for name in uploaded if name.endswith(".zip")), None)
assert source, "Upload gpu-serving-source.zip first"
with zipfile.ZipFile(source) as z:
    assert "source-manifest.json" in z.namelist(), "This is not the source bundle"
    for name in z.namelist():
        assert (repo / name).resolve().is_relative_to(repo.resolve()), "Invalid archive path"
    z.extractall(repo)
for name, digest in json.loads((repo / "source-manifest.json").read_text()).items():
    assert hashlib.sha256((repo / name).read_bytes()).hexdigest() == digest
os.chdir(repo)
os.environ["USE_TF"] = "0"
subprocess.run([sys.executable, "-m", "pip", "install", "virtualenv>=20.26,<21"], check=True)
subprocess.run([sys.executable, "-m", "virtualenv", "--system-site-packages", "--no-download", ".colab-venv"], check=True)
python = str(repo / ".colab-venv/bin/python")
subprocess.run([python, "-m", "pip", "install", "-e", ".[serving,traces]"], check=True)


In [ ]:
subprocess.run([python, "-c", "import torch; assert torch.cuda.is_available(), 'Select GPU runtime'; print(torch.cuda.get_device_name())"], check=True)
subprocess.run([python, "-m", "unittest", "discover", "-s", "tests", "-v"], check=True)
# Freeze the previously trained predictor; do not retrain on the public trace.
predictor = str(repo / "deploy/models/predictor.json")
assert Path(predictor).exists()


## Use the included verified sample
No dataset network request is made here. The setup cell checks every bundled file against the source manifest; this cell also checks the metadata against its recorded provenance hash.


In [ ]:
sample_dir = repo / "sample"
assert (sample_dir / "metadata.json").exists(), "Use the updated source ZIP that includes the sample"
source_report = json.loads((sample_dir / "provenance.json").read_text())
assert hashlib.sha256((sample_dir / "metadata.json").read_bytes()).hexdigest() == source_report["metadata_sha256"]
print("Included source requests:", source_report["eligible_requests"])
print("Dataset revision:", source_report["revision"], "License:", source_report["license"])


## Run the fixed pressure pilot
Keep COUNT=24, SELECTION=uniform, BUDGET_MIB=96 and the predictor unchanged. Selection does not search for favorable policy results. LFU is included because it is a strong baseline in the structural replay. This notebook deliberately retains outcomes where learned eviction ties or loses.


In [ ]:
OFFSET = 0
COUNT = 24
SELECTION = "uniform"
OUTPUT_CAP = 64
REPEATS = 2
BUDGET_MIB = 96
from datetime import datetime, timezone
run_dir = Path("/content") / ("kv-public-pressure-" + datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S-%f"))
archive = str(run_dir) + ".zip"
payload_file = Path("/content") / (run_dir.name + "-payloads.json")
subprocess.run([python, "-m", "kv_cache.production_trace", "build", str(sample_dir), "--offset", str(OFFSET),
                "--count", str(COUNT), "--selection", SELECTION, "--output-cap", str(OUTPUT_CAP), "--output", str(payload_file)], check=True)
bundle = json.loads(payload_file.read_text())
pressure_file = Path("/content") / (run_dir.name + "-pressure.json")
subprocess.run([python, "-m", "kv_cache.trace_pressure", str(payload_file), "--budget-mib", str(BUDGET_MIB),
                "--predictor", predictor, "--output", str(pressure_file)], check=True)
pressure = json.loads(pressure_file.read_text())
assert pressure["structurally_exercises_eviction"], "This workload does not exercise eviction; stop instead of reporting policy validation"
print("Replay provenance:", json.dumps({k:v for k,v in bundle["provenance"].items() if k not in ("source", "requests")}, indent=2))
command = [python, "-u", "-m", "kv_cache.serve_eval", "--payload-file", str(payload_file), "--predictor", predictor,
           "--cache-profile", str(pressure_file), "--device", "cuda", "--repeats", str(REPEATS), "--budget-mib", str(BUDGET_MIB), "--concurrency", "1",
           "--no-profile-stages", "--logits-mode", "last", "--reference-full-logits",
           "--policies", "none", "lru", "lfu", "learned", "--copy-modes", "shared", "--output", str(run_dir)]
try:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait() != 0:
        raise RuntimeError("Evaluation failed; download diagnostics, do not bypass correctness")
    subprocess.run([python, "-m", "kv_cache.serve_summary", str(run_dir)], check=True)
finally:
    if run_dir.exists():
        import shutil
        shutil.copyfile(repo / "source-manifest.json", run_dir / "source-manifest.json")
        shutil.copyfile(sample_dir / "provenance.json", run_dir / "dataset-provenance.json")
        shutil.copyfile(pressure_file, run_dir / "structural-pressure.json")
        shutil.make_archive(str(run_dir), "zip", run_dir)
    if Path(archive).exists():
        files.download(archive)


In [ ]:
manifest = json.loads((run_dir / "manifest.json").read_text())
print("Correctness:", manifest["status"], "Actual requests/run:", manifest["actual_requests_per_run"])
p = manifest["source_provenance"]
print("Input-capped requests:", p["input_capped_requests"], "Output-capped requests:", p["output_capped_requests"])
print("Unique retained prefixes:", p["unique_retained_prefixes"])
print("Output lengths:", [r["replay_output_tokens"] for r in p["requests"]])
print(p["scope"])
if (run_dir / "comparison.json").exists():
    for row in json.loads((run_dir / "comparison.json").read_text())["groups"]:
        print(row["policy"], "paired mean-latency reductions %:", row["paired_mean_latency_reduction_percent"],
              "throughput gains %:", row["paired_throughput_gain_percent"])

pressure = json.loads((run_dir / "structural-pressure.json").read_text())
expected = {row["policy"]: row for row in pressure["policies"]}
for item in manifest["completed"]:
    row = json.loads((run_dir / (item["label"] + ".json")).read_text())
    policy = row["info"]["policy"]
    print(item["label"], "actual hits:", row["hits"], "actual evictions:", row["evictions"])
    assert row["hits"] == expected[policy]["hits"], "Investigate cache simulation/runtime discrepancy"
    assert row["evictions"] == expected[policy]["evictions"], "Investigate eviction accounting discrepancy"
print("Runtime cache counts match the structural replay. Timing conclusions still depend on the measured results.")


## Download again if needed
This cell only downloads the most recent ZIP. It does not rerun the experiment. If the runtime was reset, an undownloaded archive may no longer exist.


In [ ]:
from pathlib import Path
from google.colab import files
archives = sorted(Path("/content").glob("kv-public-pressure-*.zip"))
if archives:
    files.download(str(archives[-1]))
else:
    print("No serving archive found in this runtime.")
